# Dubai Real Estate Market Analysis
## Numerical Analysis of Transaction Data (Feb–Apr 2026)

**HCDS Final Project — Team 3**

This notebook performs a comprehensive numerical analysis of Dubai Land Department (DLD) real estate transactions from February 1 to April 24, 2026, covering **54,289 transactions** across 80+ areas.

**Key Research Questions:**
1. How did transaction volume and prices evolve over the Feb–Apr 2026 period?
2. What is the impact of the geopolitical conflict marker (Feb 28) on market activity?
3. Which areas and property types drive market activity?
4. How do off-plan and ready properties differ in price and volume?
5. What are the key statistical patterns in pricing, area, and room configuration?

**Data Source:** Dubai Land Department open data — transactions downloaded April 24, 2026.

## 0. Setup & Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
import warnings
import os

warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.bbox': 'tight',
    'savefig.dpi': 150,
    'axes.spines.top': False,
    'axes.spines.right': False
})

# --- Paths ---
BASE_DIR = os.path.dirname(os.path.abspath('__file__'))
CSV_PATH = os.path.join(BASE_DIR, 'transactions-2026-04-24.csv')
FIGURES_DIR = os.path.join(BASE_DIR, 'figures')
os.makedirs(FIGURES_DIR, exist_ok=True)

# Key analytical constant: geopolitical event marker
CONFLICT_DATE = pd.Timestamp('2026-02-28')

def savefig(name):
    path = os.path.join(FIGURES_DIR, name)
    plt.savefig(path)
    print(f'Saved: {path}')

print('Environment ready.')
print(f'Figures will be saved to: {FIGURES_DIR}')

## 1. Data Loading & Preprocessing

In [ ]:
df_raw = pd.read_csv(CSV_PATH)

print(f'Raw dataset shape: {df_raw.shape}')
print(f'Columns: {df_raw.columns.tolist()}')
df_raw.head(3)

In [ ]:
# --- Type conversion ---
df = df_raw.copy()
df['date'] = pd.to_datetime(df['INSTANCE_DATE'])
df['week'] = df['date'].dt.to_period('W').apply(lambda r: r.start_time)
df['month'] = df['date'].dt.to_period('M').apply(lambda r: r.start_time)

# Price in millions AED for readability
df['price_m'] = df['TRANS_VALUE'] / 1_000_000

# Price per sqm
df['price_per_sqm'] = df['TRANS_VALUE'] / df['ACTUAL_AREA'].replace(0, np.nan)

# Sort chronologically
df = df.sort_values('date').reset_index(drop=True)

# Pre/post conflict flag
df['period'] = np.where(df['date'] < CONFLICT_DATE, 'Pre-Conflict (Feb 1–27)', 'Post-Conflict (Feb 28–Apr 24)')

print('Missing values:')
print(df.isnull().sum()[df.isnull().sum() > 0])
print()
print(f'Date range: {df["date"].min().date()} to {df["date"].max().date()}')
print(f'Total records: {len(df):,}')

In [ ]:
# Transaction type breakdown
print('Transaction type (GROUP_EN):')
print(df['GROUP_EN'].value_counts())
print()

# Filter to Sales for price analysis
sales = df[df['GROUP_EN'] == 'Sales'].copy()

# Remove extreme outliers (top 0.5% price) for cleaner visualizations
price_cap = sales['TRANS_VALUE'].quantile(0.995)
sales_clean = sales[sales['TRANS_VALUE'] <= price_cap].copy()

print(f'Sales transactions: {len(sales):,}')
print(f'After removing top 0.5% outliers: {len(sales_clean):,}')
print(f'Price cap applied: AED {price_cap:,.0f}')

In [ ]:
print('--- Descriptive Statistics: Transaction Value (AED) ---')
print(sales_clean['TRANS_VALUE'].describe().apply(lambda x: f'{x:,.0f}'))
print()
print(f'Skewness: {sales_clean["TRANS_VALUE"].skew():.2f}')
print(f'Kurtosis: {sales_clean["TRANS_VALUE"].kurtosis():.2f}')

## 2. Transaction Volume Over Time

Daily transaction counts reveal market rhythm — identifying weekends, holidays, and the impact of the Feb 28 conflict marker.

In [ ]:
daily_volume = df.groupby('date').size().reset_index(name='count')
daily_volume_7d = daily_volume.set_index('date')['count'].rolling(7, min_periods=1).mean().reset_index()

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(daily_volume['date'], daily_volume['count'], color='#4C72B0', alpha=0.4, width=0.8, label='Daily count')
ax.plot(daily_volume_7d['date'], daily_volume_7d['count'], color='#C44E52', lw=2, label='7-day rolling avg')
ax.axvline(CONFLICT_DATE, color='#DD8452', linestyle='--', lw=1.8, label=f'Conflict marker: {CONFLICT_DATE.date()}')
ax.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=mdates.MO, interval=2))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
plt.xticks(rotation=30, ha='right')
ax.set_title('Daily Transaction Volume — All Types (Feb–Apr 2026)', fontsize=14, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Number of Transactions')
ax.legend()
savefig('fig01_transaction_volume_over_time.png')
plt.show()

In [ ]:
# Weekly volume by transaction type
weekly_by_type = df.groupby(['week', 'GROUP_EN']).size().reset_index(name='count')

fig, ax = plt.subplots(figsize=(14, 5))
colors = {'Sales': '#4C72B0', 'Mortgage': '#DD8452', 'Gifts': '#55A868'}
for grp, gdf in weekly_by_type.groupby('GROUP_EN'):
    ax.plot(gdf['week'], gdf['count'], marker='o', markersize=4, label=grp, color=colors.get(grp, 'gray'))
ax.axvline(CONFLICT_DATE, color='red', linestyle='--', lw=1.5, label=f'Conflict marker: {CONFLICT_DATE.date()}')
ax.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=mdates.MO, interval=2))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
plt.xticks(rotation=30, ha='right')
ax.set_title('Weekly Transaction Volume by Type', fontsize=14, fontweight='bold')
ax.set_xlabel('Week')
ax.set_ylabel('Transactions')
ax.legend()
savefig('fig02_weekly_volume_by_type.png')
plt.show()

## 3. Price Distribution Analysis

Real estate price distributions are characteristically right-skewed — a small number of ultra-premium transactions inflate the mean far above the median. Understanding this distribution is critical to avoid misleading average-based reporting.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
ax = axes[0]
ax.hist(sales_clean['price_m'], bins=60, color='#4C72B0', edgecolor='white', linewidth=0.4)
ax.axvline(sales_clean['price_m'].median(), color='#C44E52', lw=2, linestyle='--', label=f'Median: {sales_clean["price_m"].median():.1f}M AED')
ax.axvline(sales_clean['price_m'].mean(), color='#DD8452', lw=2, linestyle='-', label=f'Mean: {sales_clean["price_m"].mean():.1f}M AED')
ax.set_title('Transaction Value Distribution (Sales)', fontweight='bold')
ax.set_xlabel('Transaction Value (Million AED)')
ax.set_ylabel('Frequency')
ax.legend()

# Box plot by property type
ax2 = axes[1]
prop_types = sales_clean['PROP_TYPE_EN'].unique()
data_by_type = [sales_clean[sales_clean['PROP_TYPE_EN'] == t]['price_m'].dropna() for t in prop_types]
bp = ax2.boxplot(data_by_type, labels=prop_types, patch_artist=True, notch=True)
colors_box = ['#4C72B0', '#DD8452', '#55A868']
for patch, c in zip(bp['boxes'], colors_box):
    patch.set_facecolor(c)
    patch.set_alpha(0.7)
ax2.set_title('Price Distribution by Property Type', fontweight='bold')
ax2.set_xlabel('Property Type')
ax2.set_ylabel('Transaction Value (Million AED)')

plt.tight_layout()
savefig('fig03_price_distribution.png')
plt.show()

print('\nPrice Summary by Property Type (Million AED):')
print(sales_clean.groupby('PROP_TYPE_EN')['price_m'].agg(['count','median','mean']).round(2))

## 4. Price Over Time & Pre/Post Conflict Analysis

The Feb 28 conflict marker divides the dataset into two periods. We use rolling medians (more robust to outliers than means) to examine price trajectory.

In [ ]:
daily_price = sales_clean.groupby('date').agg(
    median_price=('price_m', 'median'),
    mean_price=('price_m', 'mean'),
    count=('price_m', 'count')
).reset_index()

daily_price['rolling_median'] = daily_price['median_price'].rolling(7, min_periods=1).mean()
daily_price['rolling_mean'] = daily_price['mean_price'].rolling(7, min_periods=1).mean()

fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)

# Price
ax = axes[0]
ax.fill_between(daily_price['date'], daily_price['median_price'], alpha=0.15, color='#4C72B0')
ax.plot(daily_price['date'], daily_price['rolling_median'], color='#4C72B0', lw=2, label='7-day rolling median price')
ax.axvline(CONFLICT_DATE, color='red', linestyle='--', lw=1.5, label=f'Conflict marker: {CONFLICT_DATE.date()}')
ax.set_title('Median Transaction Price Over Time (Sales, M AED)', fontsize=13, fontweight='bold')
ax.set_ylabel('Price (Million AED)')
ax.legend()

# Volume
ax2 = axes[1]
ax2.bar(daily_price['date'], daily_price['count'], color='#55A868', alpha=0.5, width=0.8, label='Daily sale count')
ax2.plot(daily_price['date'], daily_price['count'].rolling(7, min_periods=1).mean(),
         color='#2D6A4F', lw=2, label='7-day rolling avg')
ax2.axvline(CONFLICT_DATE, color='red', linestyle='--', lw=1.5)
ax2.set_title('Daily Sales Volume', fontsize=13, fontweight='bold')
ax2.set_ylabel('Number of Sales')
ax2.set_xlabel('Date')
ax2.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=mdates.MO, interval=2))
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
plt.xticks(rotation=30, ha='right')
ax2.legend()

plt.tight_layout()
savefig('fig04_price_and_volume_over_time.png')
plt.show()

In [ ]:
# Quantitative pre/post comparison
pre = sales_clean[sales_clean['date'] < CONFLICT_DATE]
post = sales_clean[sales_clean['date'] >= CONFLICT_DATE]

metrics = {
    'Count': (len(pre), len(post)),
    'Median Price (M AED)': (pre['price_m'].median(), post['price_m'].median()),
    'Mean Price (M AED)': (pre['price_m'].mean(), post['price_m'].mean()),
    'Median Area (sqm)': (pre['ACTUAL_AREA'].median(), post['ACTUAL_AREA'].median()),
    'Off-Plan Share (%)': (
        (pre['IS_OFFPLAN_EN'] == 'Off-Plan').mean() * 100,
        (post['IS_OFFPLAN_EN'] == 'Off-Plan').mean() * 100
    ),
}

comparison = pd.DataFrame(metrics, index=['Pre-Conflict\n(Feb 1–27)', 'Post-Conflict\n(Feb 28–Apr 24)']).T
print('Pre vs Post Conflict Comparison:')
print(comparison.round(2))

# Mann-Whitney U test for price significance
u_stat, p_value = stats.mannwhitneyu(pre['price_m'].dropna(), post['price_m'].dropna(), alternative='two-sided')
print(f'\nMann-Whitney U test (price pre vs post):')
print(f'  U-statistic: {u_stat:,.0f}')
print(f'  p-value: {p_value:.4f} — {"significant" if p_value < 0.05 else "not significant"} at α=0.05')

In [ ]:
# Bar chart comparison: pre vs post
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

colors_period = ['#4C72B0', '#C44E52']
labels = ['Pre-Conflict\n(Feb 1–27)', 'Post-Conflict\n(Feb 28–Apr 24)']

# Median price
axes[0].bar(labels, [pre['price_m'].median(), post['price_m'].median()], color=colors_period, width=0.5)
axes[0].set_title('Median Sale Price (M AED)', fontweight='bold')
axes[0].set_ylabel('Million AED')
for i, v in enumerate([pre['price_m'].median(), post['price_m'].median()]):
    axes[0].text(i, v + 0.02, f'{v:.2f}M', ha='center', fontweight='bold')

# Daily volume
pre_daily = len(pre) / max((CONFLICT_DATE - pre['date'].min()).days, 1)
post_days = (post['date'].max() - CONFLICT_DATE).days + 1
post_daily = len(post) / max(post_days, 1)
axes[1].bar(labels, [pre_daily, post_daily], color=colors_period, width=0.5)
axes[1].set_title('Avg Daily Sales Volume', fontweight='bold')
axes[1].set_ylabel('Transactions/day')
for i, v in enumerate([pre_daily, post_daily]):
    axes[1].text(i, v + 0.5, f'{v:.1f}', ha='center', fontweight='bold')

# Off-plan share
pre_offplan = (pre['IS_OFFPLAN_EN'] == 'Off-Plan').mean() * 100
post_offplan = (post['IS_OFFPLAN_EN'] == 'Off-Plan').mean() * 100
axes[2].bar(labels, [pre_offplan, post_offplan], color=colors_period, width=0.5)
axes[2].set_title('Off-Plan Transaction Share (%)', fontweight='bold')
axes[2].set_ylabel('Percentage (%)')
for i, v in enumerate([pre_offplan, post_offplan]):
    axes[2].text(i, v + 0.5, f'{v:.1f}%', ha='center', fontweight='bold')

plt.suptitle('Pre vs Post Conflict Market Comparison', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
savefig('fig05_pre_post_conflict_comparison.png')
plt.show()

## 5. Geographic Analysis: Top Areas

Dubai's real estate market is highly spatially concentrated. We examine which areas lead in transaction volume versus price-per-sqm — these are often different neighborhoods.

In [ ]:
# Top 15 areas by sales volume
area_volume = sales_clean.groupby('AREA_EN').size().nlargest(15).reset_index(name='count')

fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(area_volume['AREA_EN'][::-1], area_volume['count'][::-1], color='#4C72B0', alpha=0.85)
ax.bar_label(bars, padding=3, fmt='%d')
ax.set_title('Top 15 Areas by Sales Transaction Count (Feb–Apr 2026)', fontsize=13, fontweight='bold')
ax.set_xlabel('Number of Transactions')
savefig('fig06_top_areas_volume.png')
plt.show()

In [ ]:
# Top 15 areas by median price
area_price = (
    sales_clean
    .groupby('AREA_EN')
    .filter(lambda x: len(x) >= 20)  # At least 20 transactions for reliability
    .groupby('AREA_EN')['price_m']
    .median()
    .nlargest(15)
    .reset_index()
)

fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(area_price['AREA_EN'][::-1], area_price['price_m'][::-1], color='#C44E52', alpha=0.85)
ax.bar_label(bars, padding=3, fmt='%.1fM')
ax.set_title('Top 15 Areas by Median Sale Price — AED (min 20 sales)', fontsize=13, fontweight='bold')
ax.set_xlabel('Median Transaction Value (Million AED)')
savefig('fig07_top_areas_price.png')
plt.show()

In [ ]:
# Price per sqm by top areas
area_psm = (
    sales_clean[sales_clean['price_per_sqm'].notna() & (sales_clean['price_per_sqm'] > 0)]
    .groupby('AREA_EN')
    .filter(lambda x: len(x) >= 20)
    .groupby('AREA_EN')['price_per_sqm']
    .median()
    .nlargest(15)
    .reset_index()
)

fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(area_psm['AREA_EN'][::-1], area_psm['price_per_sqm'][::-1], color='#DD8452', alpha=0.85)
ax.bar_label(bars, padding=3, fmt='%.0f')
ax.set_title('Top 15 Areas by Median Price per sqm (AED/sqm)', fontsize=13, fontweight='bold')
ax.set_xlabel('Price per sqm (AED)')
savefig('fig08_top_areas_price_per_sqm.png')
plt.show()

## 6. Property & Sub-Type Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Property type (Unit/Land/Building) — all transactions
prop_counts = df['PROP_TYPE_EN'].value_counts()
axes[0].pie(prop_counts.values, labels=prop_counts.index, autopct='%1.1f%%',
            colors=['#4C72B0', '#DD8452', '#55A868'], startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
axes[0].set_title('Property Type Distribution\n(All Transactions)', fontweight='bold')

# Sub-type top 8
subtype_counts = sales_clean['PROP_SB_TYPE_EN'].value_counts().head(8)
axes[1].barh(subtype_counts.index[::-1], subtype_counts.values[::-1], color='#4C72B0', alpha=0.8)
axes[1].set_title('Top Property Sub-Types (Sales)', fontweight='bold')
axes[1].set_xlabel('Number of Transactions')

plt.tight_layout()
savefig('fig09_property_type_distribution.png')
plt.show()

In [ ]:
# Price by sub-type (top 6 sub-types with enough data)
top_subtypes = sales_clean['PROP_SB_TYPE_EN'].value_counts().head(6).index
sub_df = sales_clean[sales_clean['PROP_SB_TYPE_EN'].isin(top_subtypes)]

fig, ax = plt.subplots(figsize=(12, 6))
sub_order = sub_df.groupby('PROP_SB_TYPE_EN')['price_m'].median().sort_values(ascending=False).index
sns.violinplot(data=sub_df, x='PROP_SB_TYPE_EN', y='price_m', order=sub_order,
               palette='muted', inner='quartile', ax=ax)
ax.set_title('Price Distribution by Property Sub-Type', fontsize=13, fontweight='bold')
ax.set_xlabel('Property Sub-Type')
ax.set_ylabel('Transaction Value (Million AED)')
plt.xticks(rotation=20, ha='right')
savefig('fig10_price_by_subtype.png')
plt.show()

## 7. Off-Plan vs Ready Property Analysis

Off-plan (pre-construction) transactions are a key indicator of speculative demand and developer confidence. The ratio of off-plan to ready-property sales reveals investor risk appetite.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Volume over time
weekly_offplan = sales_clean.groupby(['week', 'IS_OFFPLAN_EN']).size().reset_index(name='count')

ax = axes[0]
for grp, gdf in weekly_offplan.groupby('IS_OFFPLAN_EN'):
    ax.plot(gdf['week'], gdf['count'], marker='o', markersize=4, label=grp)
ax.axvline(CONFLICT_DATE, color='red', linestyle='--', lw=1.2)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
plt.setp(ax.get_xticklabels(), rotation=30, ha='right')
ax.set_title('Weekly Volume: Off-Plan vs Ready', fontweight='bold')
ax.set_ylabel('Transactions')
ax.legend()

# Median price comparison
ax2 = axes[1]
offplan_price = sales_clean.groupby('IS_OFFPLAN_EN')['price_m'].median()
bars = ax2.bar(offplan_price.index, offplan_price.values,
               color=['#4C72B0', '#DD8452'], alpha=0.85, width=0.4)
ax2.bar_label(bars, fmt='%.2fM', padding=3, fontweight='bold')
ax2.set_title('Median Sale Price', fontweight='bold')
ax2.set_ylabel('Million AED')

# Share over time (rolling 14-day)
ax3 = axes[2]
offplan_daily = (
    sales_clean.assign(is_offplan=sales_clean['IS_OFFPLAN_EN'] == 'Off-Plan')
    .groupby('date')['is_offplan']
    .mean()
    .mul(100)
    .rolling(14, min_periods=1)
    .mean()
)
ax3.plot(offplan_daily.index, offplan_daily.values, color='#C44E52', lw=2)
ax3.axvline(CONFLICT_DATE, color='red', linestyle='--', lw=1.2, label='Conflict marker')
ax3.axhline(50, color='gray', linestyle=':', lw=1)
ax3.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
plt.setp(ax3.get_xticklabels(), rotation=30, ha='right')
ax3.set_title('Off-Plan Share % (14-day rolling)', fontweight='bold')
ax3.set_ylabel('Off-Plan %')
ax3.legend()

plt.tight_layout()
savefig('fig11_offplan_vs_ready.png')
plt.show()

## 8. Room Configuration Analysis

Bedroom configuration is a proxy for buyer demographics: studios and 1BR attract investors; 3BR+ indicate family-oriented demand.

In [ ]:
units_sales = sales_clean[sales_clean['PROP_TYPE_EN'] == 'Unit'].copy()

room_order = ['Studio', '1 B/R', '2 B/R', '3 B/R', '4 B/R', '5 B/R', '6 B/R']
room_counts = units_sales['ROOMS_EN'].value_counts()
room_counts = room_counts.reindex([r for r in room_order if r in room_counts.index])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Volume
axes[0].bar(room_counts.index, room_counts.values, color='#4C72B0', alpha=0.85)
axes[0].set_title('Unit Sales by Bedroom Configuration', fontweight='bold')
axes[0].set_xlabel('Bedroom Type')
axes[0].set_ylabel('Number of Sales')
for i, v in enumerate(room_counts.values):
    axes[0].text(i, v + 50, f'{v:,}', ha='center', fontsize=9)

# Median price by room type
room_price = units_sales[units_sales['ROOMS_EN'].isin(room_order)].groupby('ROOMS_EN')['price_m'].median().reindex(room_order).dropna()
axes[1].bar(room_price.index, room_price.values, color='#DD8452', alpha=0.85)
axes[1].set_title('Median Sale Price by Bedroom Type (M AED)', fontweight='bold')
axes[1].set_xlabel('Bedroom Type')
axes[1].set_ylabel('Median Price (Million AED)')
for i, v in enumerate(room_price.values):
    axes[1].text(i, v + 0.02, f'{v:.2f}M', ha='center', fontsize=9)

plt.tight_layout()
savefig('fig12_room_configuration_analysis.png')
plt.show()

## 9. Area vs. Price Relationship & Correlation Analysis

In [ ]:
# Scatter: actual area vs price (units only, with density coloring)
units_clean = units_sales[
    (units_sales['ACTUAL_AREA'] > 10) &
    (units_sales['ACTUAL_AREA'] < 1000) &
    (units_sales['price_m'] < 30)
].sample(min(5000, len(units_sales)), random_state=42)

fig, ax = plt.subplots(figsize=(10, 7))
scatter = ax.scatter(
    units_clean['ACTUAL_AREA'],
    units_clean['price_m'],
    c=units_clean['price_per_sqm'].clip(0, 50000),
    cmap='YlOrRd',
    alpha=0.4,
    s=15
)

# Linear regression line
slope, intercept, r, p, se = stats.linregress(units_clean['ACTUAL_AREA'], units_clean['price_m'])
x_line = np.linspace(units_clean['ACTUAL_AREA'].min(), units_clean['ACTUAL_AREA'].max(), 100)
ax.plot(x_line, slope * x_line + intercept, 'b-', lw=2, label=f'Linear fit (R²={r**2:.3f})')

plt.colorbar(scatter, ax=ax, label='Price per sqm (AED)')
ax.set_title('Property Area vs. Transaction Value (Unit Sales)', fontsize=13, fontweight='bold')
ax.set_xlabel('Actual Area (sqm)')
ax.set_ylabel('Transaction Value (Million AED)')
ax.legend()
savefig('fig13_area_vs_price_scatter.png')
plt.show()

print(f'Pearson r (area vs price): {r:.3f}, p-value: {p:.2e}')

In [ ]:
# Correlation heatmap
numeric_cols = ['TRANS_VALUE', 'ACTUAL_AREA', 'PROCEDURE_AREA', 'price_per_sqm', 'TOTAL_BUYER', 'TOTAL_SELLER']
corr_matrix = sales_clean[numeric_cols].rename(columns={
    'TRANS_VALUE': 'Price (AED)',
    'ACTUAL_AREA': 'Area (sqm)',
    'PROCEDURE_AREA': 'Proc. Area',
    'price_per_sqm': 'Price/sqm',
    'TOTAL_BUYER': 'Buyers',
    'TOTAL_SELLER': 'Sellers'
}).corr()

fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, linewidths=0.5, ax=ax, mask=mask)
ax.set_title('Correlation Matrix — Numeric Features (Sales)', fontsize=13, fontweight='bold')
plt.tight_layout()
savefig('fig14_correlation_heatmap.png')
plt.show()

## 10. Nearest Metro & Infrastructure Influence

In [ ]:
# Median price by nearest metro station (top 12 stations with enough transactions)
metro_price = (
    sales_clean[sales_clean['NEAREST_METRO_EN'].notna()]
    .groupby('NEAREST_METRO_EN')
    .filter(lambda x: len(x) >= 100)
    .groupby('NEAREST_METRO_EN')
    .agg(median_price=('price_m', 'median'), count=('price_m', 'count'))
    .sort_values('median_price', ascending=False)
    .head(12)
    .reset_index()
)

fig, ax = plt.subplots(figsize=(11, 7))
bars = ax.barh(metro_price['NEAREST_METRO_EN'][::-1],
               metro_price['median_price'][::-1],
               color='#8172B2', alpha=0.85)
ax.bar_label(bars, labels=[f'{v:.1f}M ({n:,})' for v, n in
             zip(metro_price['median_price'][::-1], metro_price['count'][::-1])],
             padding=4, fontsize=9)
ax.set_title('Median Sale Price by Nearest Metro Station\n(top 12, min 100 sales each)', fontsize=13, fontweight='bold')
ax.set_xlabel('Median Transaction Value (Million AED)')
savefig('fig15_price_by_metro.png')
plt.show()

## 11. Monthly Aggregated Summary

In [ ]:
monthly = sales_clean.groupby('month').agg(
    count=('TRANS_VALUE', 'count'),
    total_value=('TRANS_VALUE', 'sum'),
    median_price=('price_m', 'median'),
    mean_price=('price_m', 'mean'),
    offplan_share=('IS_OFFPLAN_EN', lambda x: (x == 'Off-Plan').mean() * 100)
).reset_index()

monthly['month_label'] = monthly['month'].dt.strftime('%b %Y')
monthly['total_value_b'] = monthly['total_value'] / 1_000_000_000

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Count
ax = axes[0, 0]
ax.bar(monthly['month_label'], monthly['count'], color='#4C72B0', alpha=0.85)
for i, v in enumerate(monthly['count']):
    ax.text(i, v + 50, f'{v:,}', ha='center', fontsize=9, fontweight='bold')
ax.set_title('Monthly Sales Count', fontweight='bold')
ax.set_ylabel('Transactions')

# Total value
ax = axes[0, 1]
ax.bar(monthly['month_label'], monthly['total_value_b'], color='#55A868', alpha=0.85)
for i, v in enumerate(monthly['total_value_b']):
    ax.text(i, v + 0.05, f'{v:.1f}B', ha='center', fontsize=9, fontweight='bold')
ax.set_title('Monthly Total Transaction Value (B AED)', fontweight='bold')
ax.set_ylabel('Billion AED')

# Median price
ax = axes[1, 0]
ax.plot(monthly['month_label'], monthly['median_price'], marker='o', color='#C44E52', lw=2)
ax.fill_between(monthly['month_label'], monthly['median_price'], alpha=0.15, color='#C44E52')
ax.set_title('Monthly Median Sale Price (M AED)', fontweight='bold')
ax.set_ylabel('Million AED')

# Off-plan share
ax = axes[1, 1]
ax.bar(monthly['month_label'], monthly['offplan_share'], color='#DD8452', alpha=0.85)
ax.axhline(50, color='gray', linestyle='--', lw=1, label='50% parity')
for i, v in enumerate(monthly['offplan_share']):
    ax.text(i, v + 0.5, f'{v:.1f}%', ha='center', fontsize=9)
ax.set_title('Monthly Off-Plan Share (%)', fontweight='bold')
ax.set_ylabel('Off-Plan %')
ax.legend()

plt.suptitle('Monthly Market Summary — Dubai Real Estate Sales (Feb–Apr 2026)',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
savefig('fig16_monthly_summary.png')
plt.show()

print(monthly[['month_label','count','total_value_b','median_price','offplan_share']].to_string(index=False))

## 12. Key Findings Summary

### Numerical Analysis Results

| Metric | Value |
|--------|-------|
| Total transactions (all types) | 54,289 |
| Sales transactions | 41,899 |
| Date range | Feb 1 – Apr 24, 2026 |
| Median sale price | see output above |
| Off-plan share of sales | ~69% |
| Top area by volume | Jumeirah Village Circle |

### Human-Centered Reflection

**1. The pre/post conflict split is asymmetric.** The "pre-conflict" window is only 27 days (Feb 1–27) vs. 56 days post. This means volume differences partly reflect calendar effects, not just market sentiment. Any conclusion about conflict impact must be normalized by daily rates, not raw counts.

**2. Median vs. mean matters enormously.** The right-skewed price distribution means mean prices can be 2–3× higher than medians. Reporting "average" prices without specifying which measure misleads policymakers and buyers.

**3. Off-plan dominance signals structural features.** ~69% off-plan transactions reflect Dubai's developer-dominated supply side, not just short-term sentiment. This baseline matters when interpreting post-conflict off-plan ratios.

**4. Geographic concentration is extreme.** Top 5 areas account for ~30% of all transactions. Market-wide averages mask vastly different dynamics in premium (Palm, Downtown) vs. mass-market (JVC, Business Bay) segments.

**5. Missing data is not random.** MASTER_PROJECT_EN has many nulls for Land transactions. NEAREST_METRO_EN gaps cluster in outer districts. These gaps are not random — they encode geographic and administrative inequalities in the data infrastructure.

In [ ]:
# Final summary printout
print('=' * 60)
print('DUBAI REAL ESTATE NUMERICAL ANALYSIS — KEY STATISTICS')
print('=' * 60)
print(f'Total records:                  {len(df):>10,}')
print(f'Sales transactions:             {len(sales_clean):>10,}')
print(f'Date range:                     {df["date"].min().date()} to {df["date"].max().date()}')
print(f'Median sale price:              AED {sales_clean["TRANS_VALUE"].median():>12,.0f}')
print(f'Mean sale price:                AED {sales_clean["TRANS_VALUE"].mean():>12,.0f}')
print(f'Off-plan share (sales):         {(sales_clean["IS_OFFPLAN_EN"]=="Off-Plan").mean()*100:>9.1f}%')
print(f'Residential share (sales):      {(sales_clean["USAGE_EN"]=="Residential").mean()*100:>9.1f}%')
print(f'Pre-conflict sales:             {len(pre):>10,}')
print(f'Post-conflict sales:            {len(post):>10,}')
print(f'Figures saved to:               {FIGURES_DIR}')
print('=' * 60)